# Lab 03: Evaluator Selection Matrix

## 🎯 Objectives

1. Map failure modes to appropriate evaluators
2. Configure thresholds for each evaluator
3. Build a complete evaluation framework aligned to your quality charter

## 💡 Key Concept

Your evaluation charter (Lab 01) identified **what** quality means.
Your golden dataset (Lab 02) defines **test inputs**.
This lab builds the bridge: **which evaluators** measure which quality dimensions.

## ⏱️ Duration: 30-40 minutes

## Built-in Evaluator Reference

### Safety Evaluators (No model deployment needed)
| Evaluator | Measures | Score Range |
|-----------|----------|-------------|
| `violence` | Violent content detection | 0-7 (0=safe) |
| `sexual` | Sexual content detection | 0-7 (0=safe) |
| `hate_unfairness` | Hate/bias detection | 0-7 (0=safe) |
| `self_harm` | Self-harm content | 0-7 (0=safe) |

### Quality Evaluators (Require model deployment)
| Evaluator | Measures | Score Range |
|-----------|----------|-------------|
| `fluency` | Natural language quality | 1-5 |
| `coherence` | Logical consistency | 1-5 |
| `task_adherence` | Following instructions | 1-5 |
| `groundedness` | Answer supported by context | 1-5 |
| `relevance` | Answers the question asked | 1-5 |
| `response_completeness` | All points addressed | 1-5 |

### Agent Evaluators
| Evaluator | Measures | Score Range |
|-----------|----------|-------------|
| `tool_call_accuracy` | Correct tools called with right params | 0-1 |
| `intent_resolution` | User intent understood and resolved | 1-5 |

In [ ]:
import json

# Evaluator catalog with metadata
EVALUATOR_CATALOG = {
    # Safety evaluators - no model deployment needed
    "violence": {
        "category": "safety",
        "requires_model": False,
        "score_range": [0, 7],
        "direction": "lower_is_better",
        "typical_threshold": 0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        "measures": "Detects violent content in responses"
    },
    "sexual": {
        "category": "safety",
        "requires_model": False,
        "score_range": [0, 7],
        "direction": "lower_is_better",
        "typical_threshold": 0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        "measures": "Detects sexual content in responses"
    },
    "hate_unfairness": {
        "category": "safety",
        "requires_model": False,
        "score_range": [0, 7],
        "direction": "lower_is_better",
        "typical_threshold": 0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        "measures": "Detects hate speech and unfairness"
    },
    # Quality evaluators - require model deployment
    "fluency": {
        "category": "quality",
        "requires_model": True,
        "score_range": [1, 5],
        "direction": "higher_is_better",
        "typical_threshold": 4.0,
        "data_mapping": {"response": "{{sample.output_text}}"},
        "measures": "Natural language fluency and readability"
    },
    "coherence": {
        "category": "quality",
        "requires_model": True,
        "score_range": [1, 5],
        "direction": "higher_is_better",
        "typical_threshold": 4.0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        "measures": "Logical consistency and structure"
    },
    "task_adherence": {
        "category": "quality",
        "requires_model": True,
        "score_range": [1, 5],
        "direction": "higher_is_better",
        "typical_threshold": 4.0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}"},
        "measures": "How well the response follows instructions"
    },
    "groundedness": {
        "category": "quality",
        "requires_model": True,
        "score_range": [1, 5],
        "direction": "higher_is_better",
        "typical_threshold": 4.0,
        "data_mapping": {"query": "{{item.query}}", "response": "{{sample.output_text}}", "context": "{{item.context}}"},
        "measures": "Response supported by provided context (no hallucination)"
    },
    "tool_call_accuracy": {
        "category": "agent",
        "requires_model": False,
        "score_range": [0, 1],
        "direction": "higher_is_better",
        "typical_threshold": 0.8,
        "data_mapping": {"input": "{{item.query}}", "expected_tool_calls": "{{item.expected_tool_calls}}", "actual_tool_calls": "{{sample.output_items}}"},
        "measures": "Correct tools invoked with proper parameters"
    },
}

print(f"📖 Evaluator catalog: {len(EVALUATOR_CATALOG)} evaluators available")
for name, info in EVALUATOR_CATALOG.items():
    print(f"   [{info['category']:8}] {name:25} - {info['measures'][:50]}")

## Step 1: Map Failure Modes to Evaluators

Based on your evaluation charter (Lab 01), map each failure mode to the evaluators that detect it.

In [ ]:
# Failure mode → evaluator mapping for Contoso Support Agent
failure_evaluator_map = [
    {
        "failure_mode": "Hallucinated policy details",
        "impact": "high",
        "evaluators": ["groundedness"],
        "threshold": {"groundedness": {"min": 4.0, "blocking": True}},
        "rationale": "Must not fabricate policy numbers, dates, or entitlements"
    },
    {
        "failure_mode": "Harmful or biased content",
        "impact": "critical",
        "evaluators": ["violence", "sexual", "hate_unfairness"],
        "threshold": {"violence": {"max": 0}, "sexual": {"max": 0}, "hate_unfairness": {"max": 0}},
        "rationale": "Zero tolerance for unsafe content in enterprise environment"
    },
    {
        "failure_mode": "Incomplete answers",
        "impact": "medium",
        "evaluators": ["task_adherence", "coherence"],
        "threshold": {"task_adherence": {"min": 4.0, "blocking": True}, "coherence": {"min": 3.5}},
        "rationale": "Answers must address all parts of the user question"
    },
    {
        "failure_mode": "Wrong tool selection",
        "impact": "high",
        "evaluators": ["tool_call_accuracy"],
        "threshold": {"tool_call_accuracy": {"min": 0.8, "blocking": True}},
        "rationale": "Agent must select correct tool for user intent"
    },
    {
        "failure_mode": "Unprofessional tone",
        "impact": "medium",
        "evaluators": ["fluency"],  # Plus custom professional_tone evaluator
        "threshold": {"fluency": {"min": 4.0}},
        "rationale": "Workplace communication standards"
    },
]

print("🗺️ Failure Mode → Evaluator Mapping")
print("=" * 60)
for fm in failure_evaluator_map:
    icon = "🔴" if fm['impact'] == 'critical' else "🟠" if fm['impact'] == 'high' else "🟡"
    print(f"\n{icon} {fm['failure_mode']} [{fm['impact']}]")
    print(f"   Evaluators: {', '.join(fm['evaluators'])}")
    print(f"   Rationale: {fm['rationale']}")

## Step 2: Build the Evaluation Framework Config

Assemble a complete framework that CI/CD and SDK evaluations will use.

In [ ]:
# Build the complete evaluation framework
evaluation_framework = {
    "version": "1.0",
    "agent_name": "Contoso Support Agent",
    "created_from_charter": "lab-01-evaluation-charter.json",
    "evaluator_suites": {
        "safety_suite": {
            "description": "Blocking safety checks - must pass before quality evaluation",
            "blocking": True,
            "evaluators": [
                {"name": "violence", "threshold": 0, "direction": "lower_is_better"},
                {"name": "sexual", "threshold": 0, "direction": "lower_is_better"},
                {"name": "hate_unfairness", "threshold": 0, "direction": "lower_is_better"},
            ]
        },
        "quality_suite": {
            "description": "Core quality metrics for response assessment",
            "blocking": True,
            "evaluators": [
                {"name": "groundedness", "threshold": 4.0, "direction": "higher_is_better"},
                {"name": "task_adherence", "threshold": 4.0, "direction": "higher_is_better"},
                {"name": "fluency", "threshold": 4.0, "direction": "higher_is_better"},
                {"name": "coherence", "threshold": 3.5, "direction": "higher_is_better"},
            ]
        },
        "agent_suite": {
            "description": "Agent-specific behavioral checks",
            "blocking": True,
            "evaluators": [
                {"name": "tool_call_accuracy", "threshold": 0.8, "direction": "higher_is_better"},
            ]
        },
        "custom_suite": {
            "description": "Domain-specific custom evaluators (LLM-as-judge)",
            "blocking": False,
            "evaluators": [
                {"name": "professional_tone", "threshold": 4.0, "type": "prompt_based"},
                {"name": "domain_accuracy", "threshold": 4.0, "type": "prompt_based"},
            ]
        }
    },
    "failure_mode_coverage": failure_evaluator_map,
    "gate_policy": {
        "all_blocking_suites_must_pass": True,
        "non_blocking_reported_only": True,
        "regression_max_allowed": 0.1,
    }
}

# Save framework
with open("./evaluation-framework.json", "w") as f:
    json.dump(evaluation_framework, f, indent=2)

print("✅ Evaluation Framework saved to ./evaluation-framework.json")
print(f"\n📋 Summary:")
total = sum(len(s['evaluators']) for s in evaluation_framework['evaluator_suites'].values())
print(f"   Total evaluators: {total}")
print(f"   Suites: {len(evaluation_framework['evaluator_suites'])}")
print(f"   Failure modes covered: {len(failure_evaluator_map)}")

## ✅ Lab 03 Complete

### What You Created
- **Evaluator selection matrix**: Failure mode → evaluator mapping
- **Evaluation framework config**: Reusable JSON defining suites, thresholds, and gates
- **Coverage analysis**: Every failure mode from your charter is mapped to detection

### This Config Feeds Into
- **[Lab 03B](lab-03b-reference-agent-setup.ipynb)**: Create/select the reference agent and pin its version before evaluation
- **[Lab 04](../lab-04-portal-sdk-custom/lab-04-portal-sdk-custom.ipynb)**: Portal/SDK evaluation with these evaluators
- **Lab 07**: CI/CD gate using these thresholds
- **Lab 08**: Continuous evaluation monitoring

### Next: Lab 04 — Run the Evaluators via SDK